## Preliminary Model Selection: A Tiered Evaluation of Creator Nodes 🔬

The TrendTracker architecture orchestrates 15 specialized sub-agents. Many of these operate within parallelized sub-graphs (e.g., Industry_Research and Academic_Research), where the number of concurrent nodes scales dynamically based on the research topic complexity.

To optimize for both **performance** and **inference cost**, I have stratified these 15 agents into three operational tiers based on task complexity:

| Tier | Operational Focus | Description | Example Nodes |
| :---: | :--- | :--- | :--- |
| **1** | Frontier Reasoning | Requires strategic planning, high-stakes synthesis, and long-context coherence. | `research_planner`, `write_tech_blog`, `review_tech_blog` |
| **2** | Logic & Validation | Manages architectural flow control, quality gates, and multi-step query refinement. | `plan_reviewer`, `create_arxiv_queries`, `create_analysts`, etc. |
| **3** | Utility & Extraction | High-volume, narrow-task nodes. | `web_search_query`, `generate_answer`, `write_interview_memos`, etc. |

🎯 **Goal:**     
Conduct a **preliminary model selection** via LangSmith to identify the most suitable models for each tier.  

🔎 **The Problem:**     
Using Tier 1 models for high-volume Tier 3 tasks is too expensive and unneccessary; Conversely, using Tier 3 models for planning can lead to hallucinated research paths.

🔑 **The Method:**   
A cascading LLM-as-Judge framework. While each tier contains multiple roles, I selected 1–2 "creator" roles per tier for this evaluation, as their outputs are more objective to judge than "reviewer" roles:

* **Tier 1** (`research_planner`): Evaluated as the critical entry point of the agent; a valid research plan is essential for downstream success.
* **Tier 2** (`create_arxiv_query`, `create_analysts`): `create_arxiv_query` was chosen due to its strict syntax requirements, making it prone to errors. `create_analysts` was added to resolve a performance tie observed in the ArXiv query evaluation.
* Based on these results, **Tier 3** is assigned a lightweight model from the winning model family (e.g., GPT-5-mini).

### Imports & Initialization

In [ ]:
import os
from dotenv import load_dotenv
from langsmith import Client, evaluate
from pydantic import BaseModel, Field
from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage, HumanMessage

In [6]:
# Setup environment variables
load_dotenv()

# Initialize the LangSmith client
client = Client()

### Dataset Creation: A Cascading Evaluation Strategy

1. **The Seed Dataset (Tier 1 Focus)**  
    I curated 10 AI/ML research topics with varying technical complexity and time windows. This "Seed Dataset" serves two primary purposes:

    * **Unit Testing:** Validating the Tier 1 `research_planner` node's ability to decompose broad research topics into strategic research angles.

    * **End-to-End Evaluation:** Testing the full TrendTracker agent.

2. **The Derived Dataset (Tier 2 Focus)**    
    Once the `research_planner` generates its outputs, I use an LLM-as-Judge to choose it's preferred "research angles" based on certain criteria. These preferred outputs are then collected into two specialized datasets for evaluating Tier 2 nodes (e.g., `create_arxiv_query`, `create_analysts`).


In [ ]:
# Example research topics 
example_topics = [
    'Track the trend of parameter-efficient fine-tuning (PEFT) for LLMs in the last 12 months.',
    'Track the trend of Vision-Language-Action (VLA) models for robotic manipulation in the last 24 months.',
    'Track the trend of agentic workflows and multi-agent orchestration (e.g., LangGraph, CrewAI) in the last 18 months.',
    'Track the trend of synthetic data generation for training frontier models in the last 12 months.',
    'Track the trend of Mixture-of-Experts (MoE) architectures for open-source LLMs in the last year.',
    'Track the trend of speculative decoding for low-latency inference in the last 6 months.',
    'Track the trend of Retrieval-Augmented Generation (RAG) adoption in the healthcare industry over the past year.',
    'Track the trend of "Agentic RAG" architectures and technology in the last 24 months.',
    'Analyze the evolution of Small Language Models (SLMs) for on-device AI applications in the last 6 months.',
    'Track the trend of long-context window management (e.g., Ring Attention) in the last 8 months.'
]

inputs = [{'topic': topic} for topic in example_topics]

# Create a research topics dataset for evaluation
topics_dataset = client.create_dataset(
    dataset_name="ResearchTopics_TrendTracker",
    description="A curated collection of AI/ML research topics used for end-to-end evaluation of the TrendTracker agent and unit testing of the initial research_planner node."
)

# Create examples in the dataset
client.create_examples(
    dataset_id=topics_dataset.id, 
    inputs=inputs
)

{'example_ids': ['9db56bbf-8d55-40ee-b34d-e98d30140ea5',
  'b0346a95-435e-4f50-a791-5b76771910a3',
  '088602ab-994f-4823-ae21-4136d8cc215d',
  '3e9a36b8-b5c6-4f6a-bcc7-d9c0688b626f',
  'f59be32b-0126-47e1-93a8-137cac4b44f3',
  '66f551b4-27c5-4871-aa7a-2407fc1fa8d7',
  'e0a8d229-5594-4af2-b74c-4285e8e3b975',
  '4d219a9a-e374-4901-8cfd-da2c59961e63',
  '2d553755-ebb3-49d4-93f6-fc54593d0427',
  'acc1ca64-248c-44ed-babb-cce0e7ff4a72'],
 'count': 10,
 'as_of': '2026-02-28T00:23:19.346156177Z'}

In [ ]:
# Create a academic angles dataset from the LLM-as-Judge preferred outputs 
academic_angles_dataset = client.create_dataset(
    dataset_name="AcademicAngles_TrendTracker",
    description="Curated academic angles selected by LLM-as-Judge from research planner outputs for unit testing of create_arxiv_query node."
)

# Create a industry angles dataset from the LLM-as-Judge preferred outputs 
industry_angles_dataset = client.create_dataset(
    dataset_name="IndustryAngles_TrendTracker",
    description="Curated industry angles selected by LLM-as-Judge from research planner outputs for unit testing of create_analysts node."
)

# Note: Inputs of these two datasets will be manually added via LangSmith UI following the research_planner evaluation.

### LLM-as-Judge

I utilize **gpt-5.2** as the evaluator model for pairwise experiments within LangSmith. While the prompts and evaluation functions vary across different tests, the underlying output schema remains consistent to ensure standardized scoring.

In this section, I define the structured output schema and instantiate the judge:

In [89]:
# Output schema for preference
class Preference(BaseModel):
    preferred_output: int = Field(
        description="1 if the output of model_1 is better based on the rubrics, 2 if the output of model_2 is better."
        )

# LLM-as-Judge
llm_as_judge = ChatOpenAI(model="gpt-5.2").with_structured_output(Preference)

### Tier 1 Model Selection

**Model Candidates:** gpt-5.1, gpt-5.2, gemini-3.1-pro-medium, gemini-3.1-pro-high    
**Evaluation Node:** `research_planner`

The LLM-as-Judge prompt is derived from the `plan_reviewer` node in the TrendTracker agent. This ensures the evaluation identifies a model capable of generating valid research angles on the first attempt, minimizing expensive multi-turn correction cycles.

In [ ]:
# Prompt for LLM-as-Judge to evaluate research plans
plan_preference_system = """
You are a Senior Research Director specializing in AI/ML trend analysis. You are an expert at comparing multiple research plans 
to determine which is most technically sound, compliant with constraints, and ready for agent execution.

Each research plan consists of multiple research angles. Each angle must belong to a category: 'academic' or 'industry'.

<Rubric>
A good research plan:
- Strictly follows the "2-Angle Cap" for academic research (max 2 academic angles).
- Aligns the total number of angles to topic complexity:
    - Simple (narrow topics): 2 angles total.
    - Moderate (established fields with multiple sub-niches): Max 4 angles total.
    - Complex (emerging, broad, or interdisciplinary topics): Max 6 angles total.
- Contains zero search syntax (e.g., no 'cat:', 'ti:', 'abs:', or 'au:') in the descriptions.
- Ensures distinct, non-overlapping angles within the same category.
- Consists of at least one academic and one industry perspective.
Note: Cross-category complementarity is encouraged provided the perspective is strictly different. For example:
    - Angle 1 (academic): "Algorithmic advancements in Low-Rank Adaptation (LoRA)."
    - Angle 2 (industry): "Enterprise adoption challenges and cost-benefits of LoRA in production." 
    *Reasoning:* One covers 'How it works' , the other covers 'How it sells/fails'.

A bad research plan:
- Exceeds the maximum angle count for the given complexity.
- Includes forbidden search syntax or query language in the text.
- Contains redundant angles within the same category.
- Fails to provide a mix of both academic and industry sources.
</Rubric>

<Instructions>
1. Analyze the 'input' topic to determine its complexity level.
2. Audit each 'output' against the Rubric's specific pillars: Complexity Alignment (Angle Counts), 2-Angle Academic Cap, Syntax Guardrails, and Redundancy Audit.
3. Return the index of the preferred output: 1 if the output of model_1 is better, 2 if the output of model_2 is better.
</Instructions>

<Reminder>
- Avoid any position biases and ensure that the order in which the responses were presented does not influence your decision. Focus strictly on adherence to the Audit Pillars. Be as objective as possible.
- If results are equal, choose the output that is more logically sound and comprehensive, specifically favoring plans that balance academic and industry angles with the most concise and clear articulation.
</Reminder>
"""

plan_preference_human = """
Please rank these outputs according to the above instructions:

<example>
<input>
{topic}
</input>

<output>

<model_1>
{output_1}
</model_1>

<model_2>
{output_2}
</model_2>

</output>
</example>
"""

In [ ]:
# Evaluation function for research plan preference
def plan_preference(inputs: dict, outputs: list[dict]) -> list[int]:
    human_message = plan_preference_human.format(
        topic=inputs['topic'],
        output_1=outputs[0]['research_angles'],
        output_2=outputs[1]['research_angles'],    
    )

    response = llm_as_judge.invoke([SystemMessage(content=plan_preference_system), HumanMessage(content=human_message)])
    
    if response.preferred_output == 1:
        scores = [1, 0]
    else:
        scores = [0, 1]
    
    return scores

#### Evaluation Results: Plan Generation

1. **gpt-5.2 vs. gemini-3.1-pro-high:** The LLM-as-Judge favored **gpt-5.2** over gemini-3.1-pro-high (6 to 4). Notably, the latency for gemini-3.1-pro-high (26.96s) was over 3x that of gpt-5.2 (8.41s).

2. **gpt-5.1 vs. gemini-3.1-pro-medium:** The LLM-as-Judge favored **gpt-5.1** over gemini-3.1-pro-medium (7 to 3). The latency for the gemini-3.1-pro-medium (21.73s) was significantly higher than gpt-5.1 (8.53s).

3. **gpt-5.1 vs. gpt-5.2:** The LLM-as-Judge favored **gpt-5.2** over gpt-5.1 (8 to 2). While latencies are comparable, the cost of gpt-5.2 is slightly higher. 
</br>
</br>

**Pairwise Experiments**  

![research_planner_experiments](./eval_results/research_planner_experiments.png)  
</br>

**gpt-5.1(A) vs. gpt-5.2(B)**  

![research_planner::gpt-5.1 vs gpt-5.2](./eval_results/research_planner_gpt-5-1_vs_gpt-5-2.png)
</br>
</br>

#### Tier 1 Winner 

Based on these preliminary evaluation results, **gpt-5.2** is selected as the Tier 1 model for the current architecture.

### Tier 2 Model Selection  

**Model Candidates:** gpt-5, gpt-5.1, gemini-3-flash-high   
**Evaluation Node:** `create_arxiv_query`, `create_analysts`  

Following the same logic as Tier 1, the LLM-as-Judge prompts are aligned with the `review_arxiv_query` and `review_analysts` nodes. This ensures the evaluation identifies a model capable of generating valid ArXiv search queries and analysts on the first attempt, minimizing expensive multi-turn correction cycles.

In [ ]:
# Prompt for LLM-as-Judge to evaluate ArXiv query sets
query_preference_system = """
You are a Senior Research Director specializing in ArXiv API syntax, Boolean logic optimization, and multi-agent Trend Tracking architectures.
Your task is to compare two sets of ArXiv search queries generated by different models to determine which set is more "API-Ready" and research-aligned.

<Rubric>
Each query MUST follow one of the allowed formats:
   - (CATEGORY) AND (ANCHOR) AND (PIVOT_1) AND submittedDate:[START TO END]  
   - (CATEGORY) AND (ANCHOR) AND (PIVOT_1) AND (PIVOT_2) AND submittedDate:[START TO END]  
   - (CATEGORY) AND (ANCHOR) AND (PIVOT_1) AND (PIVOT_2) AND (PIVOT_3) AND submittedDate:[START TO END] 
Where:
   - CATEGORY: one or more relevant arXiv categories
   - ANCHOR: the broad domain or core technology
   - PIVOT: progressively more specific methods, aspects, or applications

A GOOD query set:
- Strictly follows Hierarchy: Flows from (CATEGORY) to (ANCHOR)to (PIVOTS). Each subsequent block narrows the search.
- Logical Decomposition: ANCHOR and PIVOT keywords are technically sound decompositions of the research angle, narrowing from broad core technology to specific methods.
- Block Constraint: Uses exactly 1 ANCHOR and no more than 3 PIVOT blocks.
- API Safety: Each query is a single line, starts immediately with `(cat:...)`, and end immediately after the closing date bracket `]`.
- Syntactic Precision: Every keyword/phrase uses both `ti:` and `abs:` fields. Multi-word phrases are wrapped in double quotes.
- Temporal Integrity: Date format is strictly `submittedDate:[YYYYMMDD0000 TO YYYYMMDD2359]` and matches the timeframe relative to current datetime.
- 1:1 Mapping: Exactly one query exists for every provided research angle.

A BAD query set:
- Semantic Disconnection: Uses keywords that are neither present in nor logical technical components of the research angle.
- Block Merging/Nesting: Combines ANCHOR and PIVOT logic into a single block or uses nested parentheses.
- Formatting Noise: Includes preamble or trailing notes/explanations.
- Field Omissions: Searching only `ti:` or only `abs:` for a keyword.
- Stale Dates: Fails to adjust the `submittedDate` range based on the the current datetime.
</Rubric>

<Instructions>
1. Audit each model output against the Rubric's specific pillars: Hierarchy, Block Constraints, API Safety, Syntactic Precision, Temporal Integrity, and 1:1 Mapping.
2. Return the index of the preferred output: 1 if model_1 is better, 2 if model_2 is better.
</Instructions>

<Reminder>
- Ignore any backslash-escaped quotes (e.g., \\" or \") as these will be handled by post-processing. Focus strictly on the query structure and research alignment.
- Avoid position and length bias. Ensure the order and length of presentation does not influence the choice.
- If both outputs are technically compliant:
   - Prefers queries that use fewer AND blocks (maximum 6 total) to avoid over-constraining the results.
   - Favors grouping related synonyms into a single large `OR` block rather than splitting them into multiple mandatory `AND` filters.
</Reminder>
"""

query_preference_human = """
Please rank these outputs according to the above instructions:

<example>
<input>
**Current Datetime:** {current_datetime}
**Research Topic:** {topic}
**Research Angles:** {academic_research_angles}
</input>

<output>
<model_1>
{output_1}
</model_1>

<model_2>
{output_2}
</model_2>
</output>
</example>
"""

In [ ]:
import datetime

# Evaluation function for ArXiv query preference
def query_preference(inputs: dict, outputs: list[dict]) -> list[int]:
    current_datetime = datetime.datetime.now().strftime("%Y%m%d%H%M")
    human_message = query_preference_human.format(
        current_datetime=current_datetime,
        topic=inputs['topic'],
        academic_research_angles=inputs['academic_research_angles'],
        output_1=outputs[0]['arxiv_queries'],
        output_2=outputs[1]['arxiv_queries'],    
    )
   
    response = llm_as_judge.invoke([SystemMessage(content=query_preference_system), HumanMessage(content=human_message)])
    
    if response.preferred_output == 1:
        scores = [1, 0]
    else:
        scores = [0, 1]
    
    return scores

#### Evaluation Results: ArXiv Query Generation

1. **gpt-5/5.1 vs. gemini-3-flash**: The LLM-as-Judge favored both **gpt-5** and **gpt-5.1** over gemini-3-flash with a decisive 9:1 win rate.
2. **gpt-5 vs. gpt-5.1** While gpt-5 and gpt-5.1 **tied** on quality, gpt-5.1 exhibited significantly lower latency and cost.  
</br>

**Pairwise Experiments:**

![arxiv_query_experiments](./eval_results/arxiv_query_experiments.png)   
</br>

**gpt-5 (A) vs. gpt-5.1 (B)**  

![gpt_5 vs gpt_5.1](./eval_results/arxiv_query_gpt-5_gpt-5-1.png)
</br>
</br>

**Next Step: Tie Breaking**  

A follow-up evaluation on the `create_analysts` node will be used to finalize the selection between gpt-5 and gpt-5.1.

In [119]:
# Prompt for LLM-as-Judge to evaluate Analyst Personas
analyst_preference_system = """
You are a Senior Research Director specializing in Multi-Agent AI Architectures and Industry Intelligence.
Your task is to compare two sets of Analyst Personas generated by different models to determine which team is more "Agent-Ready," and technically aligned with the provided research angles.

<Output_Format>
[
    {
    "expertise": "A specialized AI/ML domain lens",
    "name": "A unique semantic identifier in snake_case",
    "research_task": "A trend-oriented mission starting with an action verb."
    }
]
</Output_Format>

<Rubric>
A GOOD analyst set:
- Strict 1:1 Mapping: Generates exactly one analyst for every provided research angle. No merged roles or missing angles.
- Niche Granularity: The `expertise` field is highly specialized (e.g., "Triton Kernel Optimization" instead of "Machine Learning" or "Software Engineering").
- Strong Agency: The `research_task` starts with a decisive, technical action verb (e.g., "Trace," "Benchmark," "Deconstruct") focusing on evolution/trends.
- Logical Cohesion: The analyst's `expertise` qualifies them to perform their assigned `research_task`.
- Formatting Integrity: The model output is a raw JSON list of objects following the exact structure in <Output_Format>, containing no conversational filler.

A BAD analyst set:
- Mismatched Mapping: Missing angles or merging multiple angles into one analyst.
- Broad Personas: Generic `expertise` that would result in shallow, high-level research.
- Weak Tasks: The `research_task` is vague, lack technical depth, or fail to specify the "how" of the research.
- Expertise Mismatch: Assigning a persona with an irrelevant background to a technical task.
- Formatting Noise: Including preamble, postamble, or invalid JSON structures.
</Rubric>

<Instructions>
1. Audit each model output against the Rubric's specific pillars: 1:1 Mapping, Niche Granularity, Agency, Logical Cohesion, Trend-Focused, and Formatting.
2. Compare the technical "sharpness" of the personas. A more specialized persona is always preferred over a generalist.
3. Return the index of the preferred output: 1 if model_1 is better, 2 if model_2 is better.
</Instructions>

<Reminder>
- Avoid position and length bias. Ensure the order and length of presentation does not influence the choice. Focus strictly on adherence to the Audit Pillars. Be as objective as possible.
- If both outputs are technically compliant, choose the one with superior research alignment. This means favoring the model that best captures the technical nuances
 and specific industry terminology found in the provided research angles.
</Reminder>
"""

analyst_preference_human = """
Please rank these outputs according to the above instructions:

<example>
<input>
**Research Topic:** {topic}
**Research Angles:** {industry_research_angles}
</input>

<output>
<model_1>
{output_1}
</model_1>

<model_2>
{output_2}
</model_2>
</output>
</example>
"""

In [ ]:
# Evaluation function for analyst preference
def analyst_preference(inputs: dict, outputs: list[dict]) -> list[int]:
    human_message = analyst_preference_human.format(
        topic=inputs['topic'],
        industry_research_angles=inputs['industry_research_angles'],
        output_1=outputs[0]['industry_analysts'],
        output_2=outputs[1]['industry_analysts'],       
    )
    
    response = llm_as_judge.invoke([SystemMessage(content=analyst_preference_system), HumanMessage(content=human_message)])
    
    if response.preferred_output == 1:
        scores = [1, 0]
    else:
        scores = [0, 1]
    
    return scores

#### Evaluation Results: Analysts Generation

The LLM-as-Judge favored **gpt-5** over gpt-5.1 with a unanimous **10:0** win rate.

**gpt-5 (A) vs. gpt-5.1 (B)**  
</br>

![analysts_gpt5_gpt5-1](./eval_results/analysts_gpt-5_gpt-5-1.png)
</br>
</br>

#### Tier 2 Winner

Based on the evaluation results, **gpt-5** is selected as the Tier 2 model. While the models tied on the `create_arxiv_query` node and gpt-5.1 offered lower latency and cost, gpt-5 was chosen following its decisive **10:0** win in the `create_analysts` tie-breaker.  
</br> 


### Tier 3 Model Selection

Since both Tier 1 and Tier 2 winners are from the **gpt-5 family**, I have selected **gpt-5-mini** for Tier 3 nodes. As a lightweight model from the same family, it has shown consistent performance during unit testing throughout the agent-building process.   
</br> 


### Evaluation Summary

This notebook implements a preliminary model selection and establishes **gpt-5.2**, **gpt-5**, and **gpt-5-mini** as the Tier 1, Tier 2, and Tier 3 models based on the current balance of performance and inference cost. These findings serve as the foundation for the TrendTracker architecture. Ongoing work will focus on the hallucination and latency optimizations detailed in the [README.md](../README.md).